# Experiment 2 paired comparison

Mean ± one standard deviation across seeds, plus matched-seed deltas. Set `NANOGPT_EXPERIMENT2_SCALE` to `level0` or `level1`.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SCALE=os.getenv('NANOGPT_EXPERIMENT2_SCALE','level0')
STATE=Path(os.getenv('NANOGPT_EXPERIMENT2_STATE_FILE',f'/tmp/nanogpt-experiment2-{SCALE}-current-pair'))
PAIR=Path(os.getenv('NANOGPT_EXPERIMENT2_PAIR_ROOT',STATE.read_text().strip()))
BASE=PAIR/'baseline/results'; ADAPT=PAIR/'adaptive/results'
print('scale:',SCALE); print('pair:',PAIR)

def load(root, pattern, arm):
    frames=[]
    for run in sorted(root.glob(pattern)):
        p=run/'metrics.csv'
        if not p.exists(): continue
        d=pd.read_csv(p); d['seed']=int(run.name.rsplit('_',1)[1]); d['arm']=arm; frames.append(d)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

baseline=load(BASE,'adamw_seed_*','AdamW')
adaptive=load(ADAPT,'adamw_adaptive_wwpgd_seed_*','Adaptive WWPGD')
print('baseline seeds:',sorted(baseline.seed.unique()))
print('adaptive seeds:',sorted(adaptive.seed.unique()) if len(adaptive) else [])
assert len(baseline) and len(adaptive)


In [ ]:
fig,ax=plt.subplots(figsize=(12,6))
for label,d in [('AdamW',baseline),('Adaptive WWPGD',adaptive)]:
    a=d.groupby('step').val_loss.agg(['mean','std','count']).reset_index(); s=a['std'].fillna(0)
    line,=ax.plot(a.step,a['mean'],label=f"{label} (max n={int(a['count'].max())})")
    if int(a['count'].max())>1: ax.fill_between(a.step,a['mean']-s,a['mean']+s,alpha=.2,color=line.get_color())
ax.set(xlabel='optimizer step',ylabel='validation loss',title=f'Experiment 2 {SCALE}: validation loss mean ± 1 SD')
ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
matched=[]
for seed in sorted(set(baseline.seed)&set(adaptive.seed)):
    b=baseline[baseline.seed==seed][['step','val_loss','train_loss','val_accuracy']]
    a=adaptive[adaptive.seed==seed][['step','val_loss','train_loss','val_accuracy','loss_gap_vs_baseline']]
    m=b.merge(a,on='step',suffixes=('_adamw','_adaptive')); m['seed']=seed
    m['val_loss_delta']=m.val_loss_adaptive-m.val_loss_adamw
    m['train_loss_delta']=m.train_loss_adaptive-m.train_loss_adamw
    m['accuracy_delta_pp']=100*(m.val_accuracy_adaptive-m.val_accuracy_adamw)
    matched.append(m)
matched=pd.concat(matched,ignore_index=True)
latest=matched.sort_values('step').groupby('seed',as_index=False).tail(1)
display(latest[['seed','step','val_loss_adamw','val_loss_adaptive','val_loss_delta','accuracy_delta_pp']])
print('mean paired endpoint delta:',latest.val_loss_delta.mean())
print('SD paired endpoint delta:',latest.val_loss_delta.std(ddof=1))


In [ ]:
fig,ax=plt.subplots(figsize=(12,5))
for seed,g in matched.groupby('seed'): ax.plot(g.step,g.val_loss_delta,label=f'seed {seed}')
ax.axhline(0,linestyle='--',linewidth=1)
ax.set(xlabel='optimizer step',ylabel='adaptive − AdamW validation loss',title=f'Experiment 2 {SCALE}: matched-seed delta')
ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
selected=[]
for arm,root,pattern in [('adamw',BASE,'adamw_seed_*'),('adaptive',ADAPT,'adamw_adaptive_wwpgd_seed_*')]:
    for run in root.glob(pattern):
        p=run/'selected_checkpoint_metrics.json'
        if p.exists():
            x=json.loads(p.read_text()); x.update(arm=arm,seed=int(run.name.rsplit('_',1)[1])); selected.append(x)
selected=pd.DataFrame(selected)
pivot=selected.pivot(index='seed',columns='arm',values='test_loss').dropna()
pivot['adaptive_minus_adamw']=pivot.adaptive-pivot.adamw
display(pivot)
